In [0]:

BASE_DIR = '/Volumes/sales/landing/sales/Orders/'
CHECKPOINT_DIR = '/Volumes/sales/landing/sales/CheckPoint_Directory/'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/Orders'
CATALOG = 'Sales'
SCHEMA = 'Bronze'
TARGET_TABLE = 'Orders'
print(f'checkpoint_location : {CHECKPOINT_LOCATION}')
print(f'BASE_LOCATION = {BASE_DIR}')

In [0]:
Orders_schema = '''order_id STRING,
                   customer_id STRING, 
                   order_status STRING, 
                   order_purchase_timestamp TIMESTAMP_NTZ, 
                   order_approved_at TIMESTAMP_NTZ, 
                   order_delivered_carrier_date TIMESTAMP_NTZ, 
                   order_delivered_customer_date TIMESTAMP_NTZ, 
                   order_estimated_delivery_date TIMESTAMP_NTZ'''

In [0]:
Orders_df = spark.read.format('csv')\
                  .option('header', 'true')\
                  .schema(Orders_schema)\
                  .load(f'{BASE_DIR}')
display(Orders_df.limit(3))
Orders_df.printSchema()

#### LOAD THE `Orders` DATA INTO BRONZE LAYER
> Order_Sales . Bronze . printSchema

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
Orders_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(Orders_schema)
                 .load(f'{BASE_DIR}')
        )
Orders_trans_df = Orders_df.select(
    "order_id", 
    "customer_id", 
    "order_status", 
    "order_purchase_timestamp", 
    "order_approved_at", 
    "order_delivered_carrier_date", 
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

#Orders_trans_df.printSchema()
#Orders_trans_df.limit(3).display(checkpointLocation=f'{CHECKPOINT_DIR}/Orders_display')

In [0]:
qry = (
     Orders_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()